In [2]:
import pingouin as pg
import itertools
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc
from tqdm import tqdm
import csv
import gzip
import glob

In [3]:
def print_full(x):
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    pd.set_option('display.float_format', '{:20,.2f}'.format)
    pd.set_option('display.max_colwidth', None)
    print(x)
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.float_format')
    pd.reset_option('display.max_colwidth')

def display_full(x):
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    pd.set_option('display.float_format', '{:20,.2f}'.format)
    pd.set_option('display.max_colwidth', None)
    display(x)
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')
    pd.reset_option('display.float_format')
    pd.reset_option('display.max_colwidth')

In [4]:
def to_auc_dict(data, print_best_fpr = False):
 auc_dict = {}
 for detector in selected:
  temp = data.copy()#[data.split == 'test'].reset_index(drop=True)
  labels = [1 if (x == "machine") or (str(x) == "1") else 0 for x in temp.label]
  temp = temp.dropna()
  temp['prediction_probs'] = temp[detector].fillna(0.0).replace(-np.inf, np.finfo('float16').min).replace(np.inf, np.finfo('float16').max)
  fpr, tpr, thresholds = roc_curve(labels, temp['prediction_probs'])
  if len(pd.Series(labels).value_counts()) < 2:
     auc_dict[detector] = {'auc': -1, 'th_optim': -1, 'tpr_1%fpr': -1, 'tpr_5%fpr': -1, 'tpr_10%fpr': -1, 'tpr_20%fpr': -1, 'th_1%fpr': -1, 'th_3%fpr': -1, 'th_5%fpr': -1, 'th_10%fpr': -1, 'th_15%fpr': -1, 'th_20%fpr': -1, 'th_25%fpr': -1, 'th_30%fpr': -1, 'th_40%fpr': -1, 'th_50%fpr': -1}
  else:
     auc_dict[detector] = {'auc': auc(fpr, tpr), 'th_optim': thresholds[np.argmax(tpr - fpr)], 'tpr_1%fpr': tpr[fpr <= 0.01][-1], 'tpr_5%fpr': tpr[fpr <= 0.05][-1], 'th_1%fpr': thresholds[fpr <= 0.01][-1], 'th_3%fpr': thresholds[fpr <= 0.03][-1], 'th_5%fpr': thresholds[fpr <= 0.05][-1], 'th_10%fpr': thresholds[fpr <= 0.10][-1], 'th_15%fpr': thresholds[fpr <= 0.15][-1], 'th_20%fpr': thresholds[fpr <= 0.20][-1], 'th_25%fpr': thresholds[fpr <= 0.25][-1], 'th_30%fpr': thresholds[fpr <= 0.30][-1], 'th_40%fpr': thresholds[fpr <= 0.40][-1], 'th_50%fpr': thresholds[fpr <= 0.50][-1]}
  fpr_fully = len(temp[(temp['prediction_probs'] == 1.0) & (labels == 0)]) / len(temp[(temp['label'] == 0)])
  if print_best_fpr and (fpr_fully > 0.05):
    print('Best FPR:', detector, fpr_fully)
  if 'language' in temp.columns:
   for test_language in temp.language.unique():
    temp2 = temp[temp.language == test_language].reset_index(drop=True)
    labels = [1 if (x == "machine") or (str(x) == "1") else 0 for x in temp2.label]
    fpr, tpr, thresholds = roc_curve(labels, temp2['prediction_probs'])
    if len(pd.Series(labels).value_counts()) < 2:
        auc_dict[detector][test_language] = {'auc': -1, 'th_optim': -1, 'tpr_1%fpr': -1, 'tpr_5%fpr': -1, 'tpr_10%fpr': -1, 'tpr_20%fpr': -1, 'th_1%fpr': -1, 'th_3%fpr': -1, 'th_5%fpr': -1, 'th_10%fpr': -1, 'th_15%fpr': -1, 'th_20%fpr': -1, 'th_25%fpr': -1, 'th_30%fpr': -1, 'th_40%fpr': -1, 'th_50%fpr': -1}
    else:
        auc_dict[detector][test_language] = {'auc': auc(fpr, tpr), 'th_optim': thresholds[np.argmax(tpr - fpr)], 'tpr_1%fpr': tpr[fpr <= 0.01][-1] if len(tpr[fpr <= 0.01]) > 0 else -1, 'tpr_5%fpr': tpr[fpr <= 0.05][-1], 'tpr_10%fpr': tpr[fpr <= 0.1][-1], 'tpr_20%fpr': tpr[fpr <= 0.2][-1], 'th_1%fpr': thresholds[fpr <= 0.01][-1] if len(thresholds[fpr <= 0.01]) > 0 else -1, 'th_3%fpr': thresholds[fpr <= 0.03][-1], 'th_5%fpr': thresholds[fpr <= 0.05][-1], 'th_10%fpr': thresholds[fpr <= 0.10][-1], 'th_15%fpr': thresholds[fpr <= 0.15][-1], 'th_20%fpr': thresholds[fpr <= 0.20][-1], 'th_25%fpr': thresholds[fpr <= 0.25][-1], 'th_30%fpr': thresholds[fpr <= 0.30][-1], 'th_40%fpr': thresholds[fpr <= 0.40][-1], 'th_50%fpr': thresholds[fpr <= 0.50][-1]}
    temp2['label'] = labels
    fpr_fully = len(temp2[(temp2['prediction_probs'] == 1.0) & (temp2['label'] == 0)]) / len(temp2[(temp2['label'] == 0)])
    if print_best_fpr and (fpr_fully > 0.05):
      print('Best FPR:', detector, test_language, fpr_fully)
 domains = data.domain.unique()
 auc_dict['domains'] = domains
 return auc_dict  

def to_auc_dict_generator(data, print_best_fpr = False):
 auc_dict = {}
 for detector in selected:
  temp = data.copy()#[data.split == 'test'].reset_index(drop=True)
  labels = [1 if (x == "machine") or (str(x) == "1") else 0 for x in temp.label]
  temp = temp.dropna()
  temp['prediction_probs'] = temp[detector].fillna(0.0).replace(-np.inf, np.finfo('float16').min).replace(np.inf, np.finfo('float16').max)
  fpr, tpr, thresholds = roc_curve(labels, temp['prediction_probs'])
  if len(pd.Series(labels).value_counts()) < 2:
     auc_dict[detector] = {'auc': -1, 'th_optim': -1, 'tpr_1%fpr': -1, 'tpr_5%fpr': -1, 'tpr_10%fpr': -1, 'tpr_20%fpr': -1, 'th_1%fpr': -1, 'th_3%fpr': -1, 'th_5%fpr': -1, 'th_10%fpr': -1, 'th_15%fpr': -1, 'th_20%fpr': -1, 'th_25%fpr': -1, 'th_30%fpr': -1, 'th_40%fpr': -1, 'th_50%fpr': -1}
  else:
     auc_dict[detector] = {'auc': auc(fpr, tpr), 'th_optim': thresholds[np.argmax(tpr - fpr)], 'tpr_1%fpr': tpr[fpr <= 0.01][-1], 'tpr_5%fpr': tpr[fpr <= 0.05][-1], 'th_1%fpr': thresholds[fpr <= 0.01][-1], 'th_3%fpr': thresholds[fpr <= 0.03][-1], 'th_5%fpr': thresholds[fpr <= 0.05][-1], 'th_10%fpr': thresholds[fpr <= 0.10][-1], 'th_15%fpr': thresholds[fpr <= 0.15][-1], 'th_20%fpr': thresholds[fpr <= 0.20][-1], 'th_25%fpr': thresholds[fpr <= 0.25][-1], 'th_30%fpr': thresholds[fpr <= 0.30][-1], 'th_40%fpr': thresholds[fpr <= 0.40][-1], 'th_50%fpr': thresholds[fpr <= 0.50][-1]}
  fpr_fully = len(temp[(temp['prediction_probs'] == 1.0) & (labels == 0)]) / len(temp[(temp['label'] == 0)])
  if print_best_fpr and (fpr_fully > 0.05):
    print('Best FPR:', detector, fpr_fully)
  if 'multi_label' in temp.columns:
   for generator in [x for x in temp.multi_label.unique() if x != "human"]:
    temp2 = temp[temp.multi_label.isin(["human", generator])].reset_index(drop=True)
    labels = [1 if (x == "machine") or (str(x) == "1") else 0 for x in temp2.label]
    fpr, tpr, thresholds = roc_curve(labels, temp2['prediction_probs'])
    if len(pd.Series(labels).value_counts()) < 2:
        auc_dict[detector][generator] = {'auc': -1, 'th_optim': -1, 'tpr_1%fpr': -1, 'tpr_5%fpr': -1, 'tpr_10%fpr': -1, 'tpr_20%fpr': -1, 'th_1%fpr': -1, 'th_3%fpr': -1, 'th_5%fpr': -1, 'th_10%fpr': -1, 'th_15%fpr': -1, 'th_20%fpr': -1, 'th_25%fpr': -1, 'th_30%fpr': -1, 'th_40%fpr': -1, 'th_50%fpr': -1}
    else:
        auc_dict[detector][generator] = {'auc': auc(fpr, tpr), 'th_optim': thresholds[np.argmax(tpr - fpr)], 'tpr_1%fpr': tpr[fpr <= 0.01][-1] if len(tpr[fpr <= 0.01]) > 0 else -1, 'tpr_5%fpr': tpr[fpr <= 0.05][-1], 'tpr_10%fpr': tpr[fpr <= 0.1][-1], 'tpr_20%fpr': tpr[fpr <= 0.2][-1], 'th_1%fpr': thresholds[fpr <= 0.01][-1] if len(thresholds[fpr <= 0.01]) > 0 else -1, 'th_3%fpr': thresholds[fpr <= 0.03][-1], 'th_5%fpr': thresholds[fpr <= 0.05][-1], 'th_10%fpr': thresholds[fpr <= 0.10][-1], 'th_15%fpr': thresholds[fpr <= 0.15][-1], 'th_20%fpr': thresholds[fpr <= 0.20][-1], 'th_25%fpr': thresholds[fpr <= 0.25][-1], 'th_30%fpr': thresholds[fpr <= 0.30][-1], 'th_40%fpr': thresholds[fpr <= 0.40][-1], 'th_50%fpr': thresholds[fpr <= 0.50][-1]}
    temp2['label'] = labels
    fpr_fully = len(temp2[(temp2['prediction_probs'] == 1.0) & (temp2['label'] == 0)]) / len(temp2[(temp2['label'] == 0)])
    if print_best_fpr and (fpr_fully > 0.05):
      print('Best FPR:', detector, generator, fpr_fully)
 domains = data.domain.unique()
 auc_dict['domains'] = domains
 return auc_dict

#https://github.com/scikit-learn/scikit-learn/issues/26808
def report_np(y_true, y_pred, n_classes):

    classes = np.arange(n_classes)[None, :]
    supp = classes == y_true[:, None]
    tmp = classes == y_pred[:, None]
    hits = (tmp & supp).sum(axis=0)
    pred = tmp.sum(axis=0)
    n = y_true.shape[0]

    supp = supp.sum(axis=0)
    #https://stackoverflow.com/questions/26248654/how-to-return-0-with-divide-by-zero
    #prec = hits / pred
    pred_inv = np.array([1/i if i!=0 else 0 for i in pred])
    prec = hits * pred_inv
    #prec = np.divide(hits, pred, out=np.zeros(hits.shape, dtype=float), where=pred!=0)
    #rec = hits / supp
    supp_inv = np.array([1/i if i!=0 else 0 for i in supp])
    rec = hits * supp_inv
    #rec = np.divide(hits, supp, out=np.zeros(hits.shape, dtype=float), where=supp!=0)
    balanced_acc = rec.mean()
    prec_rec = prec + rec
    prec_rec_mult = 2 * prec * rec
    #f1 = prec_rec_mult / prec_rec
    prec_rec_inv = np.array([1/i if i!=0 else 0 for i in prec_rec])
    f1 = prec_rec_mult * prec_rec_inv
    #f1 = np.divide(prec_rec_mult, prec_rec, out=np.zeros(prec_rec_mult.shape, dtype=float), where=prec_rec!=0)
    acc = hits.sum() / n
    stacked = np.vstack([prec, rec, f1])
    macro = stacked.mean(axis=1)
    weighted = stacked @ supp / n

    return hits, pred - hits, acc, balanced_acc, supp, prec, rec, f1 , macro, weighted

def report_todict(hits, miss, acc, balanced_acc, supp, prec, rec, f1 , macro, weighted):
  report = {}
  TN = hits[0]
  FN = miss[0]
  TP = hits[1]
  FP = miss[1]
  report['fpr'] = FP/(FP+TN) if (FP+TN) > 0 else 0
  report['fnr'] = FN/(TP+FN) if (TP+FN) > 0 else 0
  human = {}
  human['precision'] = prec[0]
  human['recall'] = rec[0]
  human['f1-score'] = f1[0]
  human['support'] = supp[0]
  report['human'] = human
  machine = {}
  machine['precision'] = prec[1]
  machine['recall'] = rec[1]
  machine['f1-score'] = f1[1]
  machine['support'] = supp[1]
  report['machine'] = machine
  report['accuracy'] = acc
  report['balanced accuracy'] = balanced_acc
  macro_avg = {}
  macro_avg['precision'] = macro[0]
  macro_avg['recall'] = macro[1]
  macro_avg['f1-score'] = macro[2]
  report['macro avg'] = macro_avg
  weighted_avg = {}
  weighted_avg['precision'] = weighted[0]
  weighted_avg['recall'] = weighted[1]
  weighted_avg['f1-score'] = weighted[2]
  report['weighted avg'] = weighted_avg
  return report 

def highlight_categories(s):
    v = s['Category']
    color = 'background-color: #b6d7a8;' if v == 'F' else 'background-color: #f9cb9c;' if v == 'S' else 'background-color: #9fc5e8;'
    return [color for v in s]

def rename_detector(detector_name):
  detector_name = (detector_name
    .lower()
    .replace('nealcly-', '')
    .replace('hello-simpleai-', '')
    .replace('andreas122001-', '')
    .replace('llmdeviation', 'LLM-Deviation')
    .replace('xlm', 'XLM')
    .replace('roberta', 'RoBERTa')
    .replace('mdeberta', 'mDeBERTa')
    .replace('bert', 'BERT')
    .replace('bloomz', 'BLOOMZ')
    .replace('llama', 'Llama')
    .replace('detection-longformer', 'Detection-Longformer')
    .replace('fastdetectgpt', 'Fast-DetectGPT')
    .replace('chinese', 'Chinese')
    .replace('chatgpt', 'ChatGPT')
    .replace('binoculars', 'Binoculars')
    .replace('gemma', 'Gemma')
    .replace('2b', '2B')
    .replace('3b', '3B')
    )
  if '_' in detector_name:
    detector_name = detector_name.split('_')[0] + f" ({detector_name.split('_')[1]})"
  return detector_name

In [4]:
finetuned = pd.read_csv('results_finetuned.csv.gz')
finetuned = finetuned[finetuned.split == 'test'].reset_index(drop=True)

In [5]:
files = glob.glob("multidomain_test_ce.csv*.csv.gz")

In [6]:
#load existing detectors
results_df = pd.DataFrame()
for file in tqdm(files):
  with gzip.open(file, 'rt') as gzf: #determine the index of last column
    reader = csv.reader(gzf, dialect=csv.excel)
    last_column_index = len(next(reader)) - 1
  if "Monoculars" in file:
    df = pd.read_csv(file)
    results_df[f"binoculars"] = pd.Series([x.replace('[','').replace(']','') for x in df['binoculars']]).astype(np.float16)
    results_df[f"fastdetectgpt"] = pd.Series([x.replace('[','').replace(']','') for x in df['fastgptdetect']]).astype(np.float16)
    results_df[f"llmdeviation"] = 1-pd.Series([x.replace('[','').replace(']','') for x in df['llmd1']]).astype(np.float16)
  else:
    df = pd.read_csv(file, usecols=[last_column_index], dtype=np.float16) #load only the last column
    model_name = file.split('/')[-1].split('.csv_')[1].replace('_threshold', '').replace('.csv.gz', '').split('_')[-1]
    results_df[f"{model_name.replace('_','-')}"] = df[df.columns[-1]].astype(np.float16)
    if "unsup-simcse-roberta-base" == model_name: results_df[model_name] = 1-results_df[model_name]
  

100%|██████████| 4/4 [00:00<00:00,  6.00it/s]


In [7]:
df = pd.read_csv('multidomain_test_ce.csv.gz')
print(df.split.value_counts())
df.drop(columns=['generated', 'text', 'topic', 'split'], inplace=True)
df = pd.concat([df, results_df], axis=1)
df = pd.concat([df, finetuned.iloc[:,6:]], axis=1)
selected = results_df.columns.to_list() + finetuned.columns[6:].to_list()

split
test    55930
Name: count, dtype: int64


In [8]:
full_test = df.copy()
golden_test = df.groupby(['domain','label','language']).apply(lambda x: x.sample(100, random_state = 0)).reset_index(drop=True)
big_test = df.groupby(['domain','label','language']).apply(lambda x: x.sample(250, random_state = 0)).reset_index(drop=True)
big_test_generators = df.groupby(['domain','multi_label','language']).apply(lambda x: x.sample(200, random_state = 0)).reset_index(drop=True)

C:\Users\User\AppData\Local\Temp\ipykernel_16088\4238334442.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  golden_test = df.groupby(['domain','label','language']).apply(lambda x: x.sample(100, random_state = 0)).reset_index(drop=True)
C:\Users\User\AppData\Local\Temp\ipykernel_16088\4238334442.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  big_test = df.groupby(['domain','label','language']).apply(lambda x:

In [13]:
pd.DataFrame(full_test.groupby(['domain', 'multi_label', 'language']).size()).unstack()

0                               
language                                cs    de   hr   hu   pl   sk   sl
domain       multi_label                                                 
news         Llama-2-70b-chat-hf       278   283  281  280  283  263  279
             Mistral-7B-Instruct-v0.2  297   297  299  298  299  298  297
             aya-101                   294   298  298  300  298  298  299
             gpt-3.5-turbo-0125        296   298  298  300  298  300  298
             human                     296   296  298  300  300  299  299
             opt-iml-max-30b           298   287  293  297  294  286  295
             v5-Eagle-7B-HF            293   276  290  290  286  288  295
             vicuna-13b                276   287  291  285  278  285  292
social_media Mistral-7B-Instruct-v0.2  803  1224  779  813  920  259  393
             aya-101                   745  1190  708  733  834  252  377
             gemini                    801  1226  785  813  920  260  394
             gpt-3.5-turbo-0125        713  1181  738  693  858  254  386
             human                     710  1158  744  688  808  257  370
             opt-iml-max-30b           706  1091  693  608  799  228  358
             v5-Eagle-7B-HF            792  1198  769  812  922  256  391
             vicuna-13b                803  1229  777  797  910  260  389

In [14]:
pd.DataFrame(golden_test.groupby(['domain', 'multi_label', 'language']).size()).unstack()

0                              
language                                cs   de   hr   hu   pl   sk   sl
domain       multi_label                                                
news         Llama-2-70b-chat-hf         7    9   10   15   14   17   17
             Mistral-7B-Instruct-v0.2   17   14    6   17   13   16   16
             aya-101                    15   19   20   14   12   12   12
             gpt-3.5-turbo-0125         18   18   19   18   17   12   18
             human                     100  100  100  100  100  100  100
             opt-iml-max-30b            13   12   16   11   16   14    8
             v5-Eagle-7B-HF             18   17   17   16   14   16   15
             vicuna-13b                 12   11   12    9   14   13   14
social_media Mistral-7B-Instruct-v0.2   20   12   13   10   13   19   10
             aya-101                    13   19   15   16   23   12    9
             gemini                     12   11   12   17   16   13   15
             gpt-3.5-turbo-0125         16   23   15   13   11   20   16
             human                     100  100  100  100  100  100  100
             opt-iml-max-30b            14   10   18    9   12   10   18
             v5-Eagle-7B-HF             13   14   16   23    8   11   16
             vicuna-13b                 12   11   11   12   17   15   16

In [15]:
pd.DataFrame(big_test.groupby(['domain', 'multi_label', 'language']).size()).unstack()

0                              
language                                cs   de   hr   hu   pl   sk   sl
domain       multi_label                                                
news         Llama-2-70b-chat-hf        27   22   31   38   36   33   33
             Mistral-7B-Instruct-v0.2   30   38   31   47   29   32   38
             aya-101                    41   34   44   31   35   38   32
             gpt-3.5-turbo-0125         40   42   42   33   42   43   40
             human                     250  250  250  250  250  250  250
             opt-iml-max-30b            31   36   35   34   35   33   27
             v5-Eagle-7B-HF             47   40   40   35   34   36   37
             vicuna-13b                 34   38   27   32   39   35   43
social_media Mistral-7B-Instruct-v0.2   41   30   31   31   30   41   32
             aya-101                    36   36   39   38   36   34   31
             gemini                     39   37   29   46   42   36   38
             gpt-3.5-turbo-0125         34   49   31   28   32   40   47
             human                     250  250  250  250  250  250  250
             opt-iml-max-30b            30   29   34   28   22   31   34
             v5-Eagle-7B-HF             42   37   45   50   41   24   34
             vicuna-13b                 28   32   41   29   47   44   34

In [16]:
pd.DataFrame(big_test_generators.groupby(['domain', 'multi_label', 'language']).size()).unstack()

0                              
language                                cs   de   hr   hu   pl   sk   sl
domain       multi_label                                                
news         Llama-2-70b-chat-hf       200  200  200  200  200  200  200
             Mistral-7B-Instruct-v0.2  200  200  200  200  200  200  200
             aya-101                   200  200  200  200  200  200  200
             gpt-3.5-turbo-0125        200  200  200  200  200  200  200
             human                     200  200  200  200  200  200  200
             opt-iml-max-30b           200  200  200  200  200  200  200
             v5-Eagle-7B-HF            200  200  200  200  200  200  200
             vicuna-13b                200  200  200  200  200  200  200
social_media Mistral-7B-Instruct-v0.2  200  200  200  200  200  200  200
             aya-101                   200  200  200  200  200  200  200
             gemini                    200  200  200  200  200  200  200
             gpt-3.5-turbo-0125        200  200  200  200  200  200  200
             human                     200  200  200  200  200  200  200
             opt-iml-max-30b           200  200  200  200  200  200  200
             v5-Eagle-7B-HF            200  200  200  200  200  200  200
             vicuna-13b                200  200  200  200  200  200  200

In [13]:
#df = golden_test.copy()
#df = full_test.copy()
df = big_test.copy()
#df = big_test_generators.copy()

In [14]:
print(sorted(df.language.unique()))
df.language.value_counts()

['cs', 'de', 'hr', 'hu', 'pl', 'sk', 'sl']


language
cs    1000
de    1000
hr    1000
hu    1000
pl    1000
sk    1000
sl    1000
Name: count, dtype: int64

In [15]:
df.multi_label.value_counts()

multi_label
human                       3500
gpt-3.5-turbo-0125           543
v5-Eagle-7B-HF               542
aya-101                      505
vicuna-13b                   503
Mistral-7B-Instruct-v0.2     481
opt-iml-max-30b              439
gemini                       267
Llama-2-70b-chat-hf          220
Name: count, dtype: int64

In [16]:
results_all = pd.DataFrame()
for dataset in [to_auc_dict(df, True)]:
 for detector in tqdm(selected):
  temp = pd.DataFrame({'Dataset': "ALL_DATA", 'Detector': detector, 'Language': '{all}', 'AUC ROC': dataset[detector]['auc'], 'TPR @ 5% FPR': dataset[detector]['tpr_5%fpr'], 'Th @ 5% FPR': dataset[detector]['th_5%fpr'], 'Th @ max(TPR-FPR)': dataset[detector]['th_optim']}, index=[0])
  results_all = pd.concat([results_all, temp])
  for test_language,val in dataset[detector].items():
    if (test_language == 'auc') or ('_' in test_language): continue
    temp = pd.DataFrame({'Dataset': test_language, 'Detector': detector, 'Language': test_language, 'AUC ROC': dataset[detector][test_language]['auc'], 'TPR @ 5% FPR': val['tpr_5%fpr']}, index=[0])
    results_all = pd.concat([results_all, temp])
results_all = results_all[results_all['Detector'].isin(selected)]
#only best of each finetuned model version
temp = results_all.pivot_table(['AUC ROC', 'TPR @ 5% FPR'], 'Detector', "Dataset").reorder_levels(axis='columns', order=[1,0]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp = temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp = temp.drop_duplicates(subset=('temp','')).drop(columns='temp')
display(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))
#only worst of each finetuned model version
temp = results_all.pivot_table(['AUC ROC', 'TPR @ 5% FPR'], 'Detector', "Dataset").reorder_levels(axis='columns', order=[1,0]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp = temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp = temp.drop_duplicates(subset=('temp',''), keep='last').drop(columns='temp')
display(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

Best FPR: andreas122001-bloomz-3b-mixed-detector sl 0.062
Best FPR: xlm-roberta-base_hr-cs de 0.092
Best FPR: xlm-roberta-base_hr-cs pl 0.06
Best FPR: xlm-roberta-base_hr-cs sk 0.094
Best FPR: xlm-roberta-base_hr-cs sl 0.204
Best FPR: xlm-roberta-base_de-hu-cs pl 0.052
Best FPR: xlm-roberta-base_de-hu-cs sk 0.06
Best FPR: xlm-roberta-base_de-hu-cs sl 0.08
Best FPR: xlm-roberta-base_de-hr-hu-cs sl 0.074
Best FPR: xlm-roberta-base_de-hr sl 0.154
Best FPR: xlm-roberta-base_de-hr-hu sk 0.052
Best FPR: xlm-roberta-base_de-hr-hu sl 0.128
Best FPR: xlm-roberta-base_pl de 0.064
Best FPR: xlm-roberta-base_de-pl-hr-hu sl 0.066
Best FPR: xlm-roberta-base_hr de 0.074
Best FPR: xlm-roberta-base_hr pl 0.06
Best FPR: xlm-roberta-base_hr sk 0.098
Best FPR: xlm-roberta-base_hr sl 0.38
Best FPR: xlm-roberta-base_hr-hu de 0.072
Best FPR: xlm-roberta-base_hr-hu pl 0.072
Best FPR: xlm-roberta-base_hr-hu sk 0.1
Best FPR: xlm-roberta-base_hr-hu sl 0.212
Best FPR: xlm-roberta-base_pl-cs de 0.052
Best FPR: xlm

100%|██████████| 130/130 [00:01<00:00, 95.48it/s]


\begin{tabular}{lrrrrrrrrrrrrrrrr}
Dataset & \multicolumn{2}{r}{ALL_DATA} & \multicolumn{2}{r}{cs} & \multicolumn{2}{r}{de} & \multicolumn{2}{r}{hr} & \multicolumn{2}{r}{hu} & \multicolumn{2}{r}{pl} & \multicolumn{2}{r}{sk} & \multicolumn{2}{r}{sl} \\
 & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR \\
Detector &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
Llama-3.2-3B_de-pl-hu & \bfseries 0.9758 & \bfseries 0.8869 & \bfseries 0.9886 & \bfseries 0.9440 & \bfseries 0.9739 & 0.8780 & \bfseries 0.9829 & \bfseries 0.9300 & \bfseries 0.9851 & \bfseries 0.9540 & \bfseries 0.9765 & \bfseries 0.8880 & \bfseries 0.9779 & 0.8880 & \bfseries 0.9638 & \bfseries 0.8240 \\
mdeberta-v3-base_de-pl-hr-cs & 0.9739 & 0.8611 & 0.9835 & 0.9360 & 0.9731 & \bfseries 0.8880 & 0.9789 & 0.9140 & 0.9821 & 0.9220 & 0.9727 & 0.8700 & 0.9693 & 0.8020 & 0.9624 &

\begin{tabular}{lrrrrrrrrrrrrrrrr}
Dataset & \multicolumn{2}{r}{ALL_DATA} & \multicolumn{2}{r}{cs} & \multicolumn{2}{r}{de} & \multicolumn{2}{r}{hr} & \multicolumn{2}{r}{hu} & \multicolumn{2}{r}{pl} & \multicolumn{2}{r}{sk} & \multicolumn{2}{r}{sl} \\
 & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR \\
Detector &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
mdeberta-v3-base_hu-cs & \bfseries 0.9563 & 0.4186 & \bfseries 0.9759 & 0.9240 & 0.9428 & 0.4420 & 0.9620 & 0.8020 & 0.9799 & 0.9280 & 0.9437 & 0.5160 & 0.9594 & 0.3940 & \bfseries 0.9309 & 0.3780 \\
Llama-3.2-3B_hr & 0.9460 & 0.0000 & 0.9723 & \bfseries 0.9400 & \bfseries 0.9608 & \bfseries 0.8080 & \bfseries 0.9721 & \bfseries 0.9200 & 0.9750 & 0.9140 & \bfseries 0.9524 & \bfseries 0.7420 & \bfseries 0.9627 & \bfseries 0.8320 & 0.8490 & 0.0000 \\
xlm-roberta-base_hr & 0.9061 & 

In [ ]:
#statistical significance of per-train-languages-combination results (mean across base models)
temp = results_all[results_all['Detector'].str.contains('_')].pivot_table('AUC ROC', 'Detector', "Dataset").sort_index(axis='columns')
temp = temp.sort_values(by='ALL_DATA', ascending=False)
temp['temp'] = ['-'.join(sorted(x.split('_')[1].split('-'))) for x in temp.index]
temp = temp[['temp', 'ALL_DATA']].copy()

#temp = temp.pivot(index=['Category', 'Detector'], columns='temp', values='ALL_DATA').reset_index()

res_df = pd.DataFrame()
significant = 0
not_significant = 0
for (src, trg) in itertools.combinations_with_replacement(temp.temp.unique(), 2):
   if src == trg: continue
   print(f"({trg}, {src})")
   res = pg.ttest(temp[temp["temp"] == trg]['ALL_DATA'], temp[temp["temp"] == src]['ALL_DATA'], paired=True)
   if res.loc['T-test', 'p-val'] < 0.05:
     significant += 1
     display(res)#.style.apply(lambda _: np.where(res['p-val'] >= 0.05, 'background-color: yellow', '')))
   else:
      not_significant += 1
print(f"significant: {significant}\n not significant: {not_significant}")

(de-pl, de-hu-pl)
(cs-de-hr-pl, de-hu-pl)
(cs-de-pl, de-hu-pl)
(cs-de-hr-hu-pl, de-hu-pl)
(pl, de-hu-pl)
(de-hr-hu-pl, de-hu-pl)


,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-3.352918,3,two-sided,0.043964,"[-0.01, -0.0]",0.604213,2.599,0.139393


(de-hu, de-hu-pl)
(cs-de, de-hu-pl)
(cs-de-hu-pl, de-hu-pl)


,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-4.457243,3,two-sided,0.021022,"[-0.01, -0.0]",0.530403,4.277,0.118945


(cs-hu-pl, de-hu-pl)


...

(cs-hr-hu-pl, hu-pl)
(hr-hu-pl, hu-pl)
(cs-hu, hu-pl)
(cs-hr, hu-pl)
(hu, hu-pl)
(hr, hu-pl)


,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-3.424284,3,two-sided,0.041712,"[-0.05, -0.0]",1.513872,2.694,0.53965


(hr-hu-pl, cs-hr-hu-pl)
(cs-hu, cs-hr-hu-pl)
(cs-hr, cs-hr-hu-pl)
(hu, cs-hr-hu-pl)
(hr, cs-hr-hu-pl)
(cs-hu, hr-hu-pl)
(cs-hr, hr-hu-pl)
(hu, hr-hu-pl)
(hr, hr-hu-pl)
(cs-hr, cs-hu)
(hu, cs-hu)
(hr, cs-hu)
(hu, cs-hr)
(hr, cs-hr)


,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-3.353521,3,two-sided,0.043944,"[-0.03, -0.0]",0.8252,2.6,0.215263


(hr, hu)
significant: 132
 not significant: 333


In [17]:
statistical = ['binoculars', 'fastdetectgpt', 'llmdeviation']
pretrained = ['andreas122001-bloomz-3b-mixed-detector', 'Hello-SimpleAI-chatgpt-detector-roberta-chinese', 'nealcly-detection-longformer']

In [32]:
df.corr(numeric_only=True).iloc[2][9:].describe()

count    124.000000
mean       0.059047
std        0.044424
min       -0.053709
25%        0.032399
50%        0.057537
75%        0.082611
max        0.197213
Name: length, dtype: float64

In [28]:
print_full(df.corr(numeric_only=True).iloc[2])

index                                                            -0.41
label                                                             0.10
length                                                            1.00
Hello-SimpleAI-chatgpt-detector-roberta-chinese                   0.17
andreas122001-bloomz-3b-mixed-detector                           -0.27
binoculars                                                        0.40
fastdetectgpt                                                     0.38
llmdeviation                                                      0.52
nealcly-detection-longformer                                      0.02
xlm-roberta-base_hr-cs                                            0.01
xlm-roberta-base_de-hu-cs                                         0.01
xlm-roberta-base_de-hr-hu-cs                                      0.03
xlm-roberta-base_de-hr                                            0.04
xlm-roberta-base_de-hr-hu                                        -0.00
xlm-ro

In [18]:
#AUC ROC of only best of each finetuned model version
temp = results_all.pivot_table('AUC ROC', 'Detector', "Dataset").sort_index(axis='columns')
temp = temp.sort_values(by='ALL_DATA', ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp['Category'] = ['S' if x in statistical else 'P' if x in pretrained else 'F' for x in temp['temp']]
temp = temp.drop_duplicates(subset='temp').drop(columns='temp')
temp = temp.reset_index()
temp['Detector'] = temp['Detector'].apply(rename_detector)
#temp = temp.set_index('Detector')
temp = temp[[temp.columns[-1]] + list(temp.columns[:-1])]
display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', subset=[x for x in temp.columns if x not in ['Category', 'Detector']], axis=0).apply(highlight_categories, axis=1)) #.hide('Category', axis=1)
print(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', subset=[x for x in temp.columns if x not in ['Category', 'Detector']],axis=0).applymap_index(lambda v: "font-weight: bold;", axis=0).applymap_index(lambda v: "font-weight: bold;", axis=1).hide().background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None).to_latex(convert_css=True))

Dataset,Category,Detector,ALL_DATA,cs,de,hr,hu,pl,sk,sl
0,F,Llama-3.2-3B (de-pl-hu),0.9758,0.9886,0.9739,0.9829,0.9851,0.9765,0.9779,0.9638
1,F,mDeBERTa-v3-base (de-pl-hr-cs),0.9739,0.9835,0.9731,0.9789,0.9821,0.9727,0.9693,0.9624
2,F,Gemma-2-2B (de-pl-cs),0.9660,0.9837,0.9697,0.9546,0.9750,0.9620,0.9765,0.9434
3,F,XLM-RoBERTa-base (de-pl-hr-hu-cs),0.9621,0.9778,0.9484,0.9744,0.9748,0.9541,0.9606,0.9497
4,S,Fast-DetectGPT,0.7904,0.7943,0.8074,0.8228,0.7587,0.7830,0.7829,0.8133
5,S,Binoculars,0.7675,0.7811,0.7924,0.8000,0.7517,0.7681,0.7496,0.7650
6,S,LLM-Deviation,0.6887,0.7543,0.6666,0.7258,0.6855,0.6991,0.7141,0.7337
7,P,BLOOMZ-3B-mixed-detector,0.6740,0.6769,0.6945,0.6690,0.6836,0.6752,0.7292,0.5997
8,P,ChatGPT-detector-RoBERTa-Chinese,0.6492,0.6045,0.7055,0.6442,0.7238,0.6361,0.6885,0.6629
9,P,Detection-Longformer,0.5629,0.5687,0.4860,0.6519,0.6319,0.5900,0.4636,0.5654


\begin{tabular}{llrrrrrrrr}
\bfseries Category & \bfseries Detector & \bfseries ALL_DATA & \bfseries cs & \bfseries de & \bfseries hr & \bfseries hu & \bfseries pl & \bfseries sk & sl \\
F & Llama-3.2-3B (de-pl-hu) & \bfseries {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9758 & \bfseries {\cellcolor[HTML]{78ABD0}} \color[HTML]{000000} 0.9886 & \bfseries {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9739 & \bfseries {\cellcolor[HTML]{7BACD1}} \color[HTML]{000000} 0.9829 & \bfseries {\cellcolor[HTML]{79ABD0}} \color[HTML]{000000} 0.9851 & \bfseries {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9765 & \bfseries {\cellcolor[HTML]{7DACD1}} \color[HTML]{000000} 0.9779 & \bfseries {\cellcolor[HTML]{83AFD3}} \color[HTML]{000000} 0.9638 \\
F & mDeBERTa-v3-base (de-pl-hr-cs) & {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9739 & {\cellcolor[HTML]{7BACD1}} \color[HTML]{000000} 0.9835 & {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9731 & {\cellcolor[HTML]{7DACD1}} \color[HTML

C:\Users\User\AppData\Local\Temp\ipykernel_16088\2770399120.py:12: FutureWarning: Styler.applymap_index has been deprecated. Use Styler.map_index instead.
  print(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', subset=[x for x in temp.columns if x not in ['Category', 'Detector']],axis=0).applymap_index(lambda v: "font-weight: bold;", axis=0).applymap_index(lambda v: "font-weight: bold;", axis=1).hide().background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None).to_latex(convert_css=True))


In [ ]:
#TPR @ 5% FPR of only best of each finetuned model version
temp = results_all.pivot_table('TPR @ 5% FPR', 'Detector', "Dataset").sort_index(axis='columns')
temp = temp.sort_values(by='ALL_DATA', ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp['Category'] = ['S' if x in statistical else 'P' if x in pretrained else 'F' for x in temp['temp']]
temp = temp.drop_duplicates(subset='temp').drop(columns='temp')
temp = temp.reset_index()
temp['Detector'] = temp['Detector'].apply(rename_detector)
#temp = temp.set_index('Detector')
temp = temp[[temp.columns[-1]] + list(temp.columns[:-1])]
display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', subset=[x for x in temp.columns if x not in ['Category', 'Detector']], axis=0).apply(highlight_categories, axis=1)) #.hide('Category', axis=1)
print(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', subset=[x for x in temp.columns if x not in ['Category', 'Detector']],axis=0).applymap_index(lambda v: "font-weight: bold;", axis=0).applymap_index(lambda v: "font-weight: bold;", axis=1).hide().background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None).to_latex(convert_css=True))

In [24]:
#mean AUC ROC of language combinations of finetuned detectors
temp = results_all[results_all['Detector'].str.contains('_')].pivot_table('AUC ROC', 'Detector', "Dataset").sort_index(axis='columns')
temp = temp.sort_values(by='ALL_DATA', ascending=False)
temp['temp'] = ['-'.join(sorted(x.split('_')[1].split('-'))) for x in temp.index]
temp = temp.groupby('temp').agg(['mean', 'std']).sort_values(by=('ALL_DATA', 'mean'), ascending=False)
display(temp.style.format(precision=3).highlight_max(props='font-weight: bold;', axis=0)) #.background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None)
print(temp.style.format(precision=3).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

\begin{tabular}{lrrrrrrrrrrrrrrrr}
Dataset & \multicolumn{2}{r}{ALL_DATA} & \multicolumn{2}{r}{cs} & \multicolumn{2}{r}{de} & \multicolumn{2}{r}{hr} & \multicolumn{2}{r}{hu} & \multicolumn{2}{r}{pl} & \multicolumn{2}{r}{sk} & \multicolumn{2}{r}{sl} \\
 & mean & std & mean & std & mean & std & mean & std & mean & std & mean & std & mean & std & mean & std \\
temp &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
cs-de-hr-hu-pl & \bfseries 0.967 & 0.005 & 0.982 & 0.003 & 0.963 & 0.010 & 0.975 & 0.002 & 0.979 & 0.004 & 0.962 & 0.007 & 0.969 & 0.005 & 0.941 & 0.016 \\
de-hu-pl & 0.966 & 0.009 & \bfseries 0.982 & 0.007 & 0.967 & 0.011 & 0.965 & 0.012 & 0.980 & 0.005 & 0.966 & 0.010 & 0.969 & 0.011 & \bfseries 0.953 & 0.014 \\
de-pl & 0.966 & 0.008 & 0.981 & 0.006 & 0.969 & 0.009 & 0.965 & 0.012 & 0.974 & 0.011 & 0.967 & 0.006 & 0.967 & 0.011 & 0.951 & 0.010 \\
cs-de & 0.966 & 0.004 & 0.980 & 0.002 & \bfseries 0.972 & 0.007 & 0.963 & 0.012 & 0.973 & 0.008 & 0.951 & 0.007 & \bfseries 0.973 &

In [20]:
#mean TPR @ 5% FPR of language combinations of finetuned detectors
temp = results_all[results_all['Detector'].str.contains('_')].pivot_table('TPR @ 5% FPR', 'Detector', "Dataset").sort_index(axis='columns')
temp = temp.sort_values(by='ALL_DATA', ascending=False)
temp['temp'] = ['-'.join(sorted(x.split('_')[1].split('-'))) for x in temp.index]
temp = temp.groupby('temp').agg(['mean', 'std']).sort_values(by=('ALL_DATA', 'mean'), ascending=False)
display(temp.style.format(precision=3).highlight_max(props='font-weight: bold;', axis=0)) #.background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None)
print(temp.style.format(precision=3).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

\begin{tabular}{lrrrrrrrrrrrrrrrr}
Dataset & \multicolumn{2}{r}{ALL_DATA} & \multicolumn{2}{r}{cs} & \multicolumn{2}{r}{de} & \multicolumn{2}{r}{hr} & \multicolumn{2}{r}{hu} & \multicolumn{2}{r}{pl} & \multicolumn{2}{r}{sk} & \multicolumn{2}{r}{sl} \\
 & mean & std & mean & std & mean & std & mean & std & mean & std & mean & std & mean & std & mean & std \\
temp &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
cs-de-hr-hu-pl & \bfseries 0.849 & 0.036 & 0.936 & 0.019 & 0.844 & 0.080 & 0.893 & 0.020 & 0.922 & 0.038 & 0.825 & 0.054 & 0.863 & 0.041 & 0.275 & 0.337 \\
de-pl & 0.844 & 0.033 & 0.916 & 0.023 & 0.863 & 0.045 & 0.850 & 0.065 & 0.901 & 0.050 & 0.849 & 0.062 & 0.845 & 0.056 & 0.731 & 0.033 \\
cs-de & 0.840 & 0.037 & 0.937 & 0.019 & \bfseries 0.879 & 0.039 & 0.855 & 0.051 & 0.893 & 0.047 & 0.762 & 0.086 & 0.855 & 0.045 & \bfseries 0.750 & 0.067 \\
cs-de-pl & 0.837 & 0.037 & 0.932 & 0.021 & 0.843 & 0.056 & 0.820 & 0.086 & 0.909 & 0.032 & 0.837 & 0.057 & 0.858 & 0.052 & 0.626 & 0.2

In [103]:
#mean AUC ROC of finetuned base models
temp = results_all[results_all['Detector'].str.contains('_')].pivot_table('AUC ROC', 'Detector', "Dataset").sort_index(axis='columns')
temp = temp.sort_values(by='ALL_DATA', ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp = temp.groupby('temp').mean().sort_values(by='ALL_DATA', ascending=False)
display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

Dataset,ALL_DATA,cs,de,hr,hu,pl,sk,sl
temp,,,,,,,,
mdeberta-v3-base,0.9661,0.9821,0.9609,0.9707,0.9812,0.9574,0.9639,0.9487
Llama-3.2-3B,0.9625,0.9805,0.9611,0.9727,0.9800,0.9627,0.9671,0.9224
gemma-2-2b,0.9486,0.9801,0.9445,0.9551,0.9718,0.9538,0.9652,0.9114
xlm-roberta-base,0.9457,0.9707,0.9214,0.9634,0.9712,0.9422,0.9434,0.9161


\begin{tabular}{lrrrrrrrr}
Dataset & ALL_DATA & cs & de & hr & hu & pl & sk & sl \\
temp &  &  &  &  &  &  &  &  \\
mdeberta-v3-base & \bfseries 0.9661 & \bfseries 0.9821 & 0.9609 & 0.9707 & \bfseries 0.9812 & 0.9574 & 0.9639 & \bfseries 0.9487 \\
Llama-3.2-3B & 0.9625 & 0.9805 & \bfseries 0.9611 & \bfseries 0.9727 & 0.9800 & \bfseries 0.9627 & \bfseries 0.9671 & 0.9224 \\
gemma-2-2b & 0.9486 & 0.9801 & 0.9445 & 0.9551 & 0.9718 & 0.9538 & 0.9652 & 0.9114 \\
xlm-roberta-base & 0.9457 & 0.9707 & 0.9214 & 0.9634 & 0.9712 & 0.9422 & 0.9434 & 0.9161 \\
\end{tabular}



In [21]:
#mean TPR @ 5% FPR of finetuned base models
temp = results_all[results_all['Detector'].str.contains('_')].pivot_table('TPR @ 5% FPR', 'Detector', "Dataset").sort_index(axis='columns')
temp = temp.sort_values(by='ALL_DATA', ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp = temp.groupby('temp').mean().sort_values(by='ALL_DATA', ascending=False)
display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

Dataset,ALL_DATA,cs,de,hr,hu,pl,sk,sl
temp,,,,,,,,
mdeberta-v3-base,0.7816,0.9286,0.7879,0.8163,0.9276,0.6664,0.7392,0.5750
gemma-2-2b,0.7432,0.9381,0.6021,0.8427,0.9148,0.8033,0.8279,0.3256
xlm-roberta-base,0.6635,0.8605,0.5652,0.8230,0.8585,0.5843,0.6014,0.3865
Llama-3.2-3B,0.6366,0.9442,0.8065,0.8740,0.9079,0.7737,0.7571,0.3492


\begin{tabular}{lrrrrrrrr}
Dataset & ALL_DATA & cs & de & hr & hu & pl & sk & sl \\
temp &  &  &  &  &  &  &  &  \\
mdeberta-v3-base & \bfseries 0.7816 & 0.9286 & 0.7879 & 0.8163 & \bfseries 0.9276 & 0.6664 & 0.7392 & \bfseries 0.5750 \\
gemma-2-2b & 0.7432 & 0.9381 & 0.6021 & 0.8427 & 0.9148 & \bfseries 0.8033 & \bfseries 0.8279 & 0.3256 \\
xlm-roberta-base & 0.6635 & 0.8605 & 0.5652 & 0.8230 & 0.8585 & 0.5843 & 0.6014 & 0.3865 \\
Llama-3.2-3B & 0.6366 & \bfseries 0.9442 & \bfseries 0.8065 & \bfseries 0.8740 & 0.9079 & 0.7737 & 0.7571 & 0.3492 \\
\end{tabular}



In [71]:
#generators
results_all = pd.DataFrame()
for dataset in [to_auc_dict_generator(df, True)]:
 for detector in tqdm(selected):
  temp = pd.DataFrame({'Dataset': "ALL_DATA", 'Detector': detector, 'Generator': '{all}', 'AUC ROC': dataset[detector]['auc'], 'TPR @ 5% FPR': dataset[detector]['tpr_5%fpr'], 'Th @ 5% FPR': dataset[detector]['th_5%fpr'], 'Th @ max(TPR-FPR)': dataset[detector]['th_optim']}, index=[0])
  results_all = pd.concat([results_all, temp])
  for generator,val in dataset[detector].items():
    if (generator == 'auc') or ('_' in generator): continue
    temp = pd.DataFrame({'Dataset': generator, 'Detector': detector, 'Generator': generator, 'AUC ROC': dataset[detector][generator]['auc'], 'TPR @ 5% FPR': val['tpr_5%fpr']}, index=[0])
    results_all = pd.concat([results_all, temp])
results_all = results_all[results_all['Detector'].isin(selected)]
#only best of each finetuned model version
temp = results_all.pivot_table(['AUC ROC', 'TPR @ 5% FPR'], 'Detector', "Dataset").reorder_levels(axis='columns', order=[1,0]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp = temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp = temp.drop_duplicates(subset=('temp','')).drop(columns='temp')
display(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))
#only worst of each finetuned model version
temp = results_all.pivot_table(['AUC ROC', 'TPR @ 5% FPR'], 'Detector', "Dataset").reorder_levels(axis='columns', order=[1,0]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp = temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp = temp.drop_duplicates(subset=('temp',''), keep='last').drop(columns='temp')
display(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

Best FPR: xlm-roberta-base_hr-cs Llama-2-70b-chat-hf 0.07857142857142857
Best FPR: xlm-roberta-base_hr-cs Mistral-7B-Instruct-v0.2 0.07857142857142857
Best FPR: xlm-roberta-base_hr-cs aya-101 0.07857142857142857
Best FPR: xlm-roberta-base_hr-cs gpt-3.5-turbo-0125 0.07857142857142857
Best FPR: xlm-roberta-base_hr-cs opt-iml-max-30b 0.07857142857142857
Best FPR: xlm-roberta-base_hr-cs v5-Eagle-7B-HF 0.07857142857142857
Best FPR: xlm-roberta-base_hr-cs vicuna-13b 0.07857142857142857
Best FPR: xlm-roberta-base_hr-cs gemini 0.07857142857142857
Best FPR: xlm-roberta-base_hr Llama-2-70b-chat-hf 0.09785714285714285
Best FPR: xlm-roberta-base_hr Mistral-7B-Instruct-v0.2 0.09785714285714285
Best FPR: xlm-roberta-base_hr aya-101 0.09785714285714285
Best FPR: xlm-roberta-base_hr gpt-3.5-turbo-0125 0.09785714285714285
Best FPR: xlm-roberta-base_hr opt-iml-max-30b 0.09785714285714285
Best FPR: xlm-roberta-base_hr v5-Eagle-7B-HF 0.09785714285714285
Best FPR: xlm-roberta-base_hr vicuna-13b 0.097857142

100%|██████████| 130/130 [00:01<00:00, 104.87it/s]


\begin{tabular}{lrrrrrrrrrrrrrrrrrr}
Dataset & \multicolumn{2}{r}{ALL_DATA} & \multicolumn{2}{r}{Llama-2-70b-chat-hf} & \multicolumn{2}{r}{Mistral-7B-Instruct-v0.2} & \multicolumn{2}{r}{aya-101} & \multicolumn{2}{r}{gemini} & \multicolumn{2}{r}{gpt-3.5-turbo-0125} & \multicolumn{2}{r}{opt-iml-max-30b} & \multicolumn{2}{r}{v5-Eagle-7B-HF} & \multicolumn{2}{r}{vicuna-13b} \\
 & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR \\
Detector &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
Llama-3.2-3B_de-pl-hu & \bfseries 0.9749 & \bfseries 0.8871 & 0.9904 & 0.9714 & 0.9688 & \bfseries 0.8371 & 0.9655 & \bfseries 0.8421 & \bfseries 0.9754 & \bfseries 0.8814 & \bfseries 0.9828 & \bfseries 0.9204 & \bfseries 0.9632 & \bfseries 0.8446 & 0.9822 & 0.9254 & 0.9789 & \bfseries 0.9136 \\
mdeberta-v3-base_de-pl-hr-hu-cs 

\begin{tabular}{lrrrrrrrrrrrrrrrrrr}
Dataset & \multicolumn{2}{r}{ALL_DATA} & \multicolumn{2}{r}{Llama-2-70b-chat-hf} & \multicolumn{2}{r}{Mistral-7B-Instruct-v0.2} & \multicolumn{2}{r}{aya-101} & \multicolumn{2}{r}{gemini} & \multicolumn{2}{r}{gpt-3.5-turbo-0125} & \multicolumn{2}{r}{opt-iml-max-30b} & \multicolumn{2}{r}{v5-Eagle-7B-HF} & \multicolumn{2}{r}{vicuna-13b} \\
 & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR \\
Detector &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
mdeberta-v3-base_hr & \bfseries 0.9566 & \bfseries 0.7557 & \bfseries 0.9833 & \bfseries 0.9529 & \bfseries 0.9591 & \bfseries 0.7518 & \bfseries 0.9450 & \bfseries 0.6689 & \bfseries 0.9547 & \bfseries 0.7750 & \bfseries 0.9620 & \bfseries 0.7654 & \bfseries 0.9242 & 0.5429 & \bfseries 0.9728 & \bfseries 0.8943 & \bfseries 0.9

In [ ]:
temp['Category'] = ['S' if x in statistical else 'P' if x in pretrained else 'F' for x in temp['temp']]
temp = temp.drop_duplicates(subset='temp').drop(columns='temp')
temp = temp.reset_index()
temp['Detector'] = temp['Detector'].apply(rename_detector)
#temp = temp.set_index('Detector')
temp = temp[[temp.columns[-1]] + list(temp.columns[:-1])]
display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', subset=[x for x in temp.columns if x not in ['Category', 'Detector']], axis=0).apply(highlight_categories, axis=1)) #.hide('Category', axis=1)


In [73]:
#AUC ROC of only best of each finetuned model version
temp = results_all.pivot_table('AUC ROC', 'Detector', "Dataset").sort_index(axis='columns')
temp = temp.sort_values(by='ALL_DATA', ascending=False)
temp['temp'] = [x.split('_')[0] for x in temp.index]
temp['Category'] = ['S' if x in statistical else 'P' if x in pretrained else 'F' for x in temp['temp']]
temp = temp.drop_duplicates(subset='temp').drop(columns='temp')
temp = temp.reset_index()
temp['Detector'] = temp['Detector'].apply(rename_detector)
temp = temp[[temp.columns[-1]] + list(temp.columns[:-1])]
display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', subset=[x for x in temp.columns if x not in ['Category', 'Detector']], axis=0).hide().background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None))
print(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', subset=[x for x in temp.columns if x not in ['Category', 'Detector']], axis=0).applymap_index(lambda v: "font-weight: bold;", axis=0).applymap_index(lambda v: "font-weight: bold;", axis=1).hide().background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None).to_latex(convert_css=True))

Category,Detector,ALL_DATA,Llama-2-70b-chat-hf,Mistral-7B-Instruct-v0.2,aya-101,gemini,gpt-3.5-turbo-0125,opt-iml-max-30b,v5-Eagle-7B-HF,vicuna-13b
F,Llama-3.2-3B (de-pl-hu),0.9749,0.9904,0.9688,0.9655,0.9754,0.9828,0.9632,0.9822,0.9789
F,mDeBERTa-v3-base (de-pl-hr-hu-cs),0.9734,0.9935,0.9704,0.9662,0.9706,0.9797,0.9505,0.9848,0.9801
F,Gemma-2-2B (de-pl-cs),0.9627,0.9777,0.9555,0.9612,0.9585,0.9717,0.9452,0.9722,0.9653
F,XLM-RoBERTa-base (de-pl-hr-hu-cs),0.9613,0.9821,0.9430,0.9526,0.9626,0.9745,0.9403,0.9732,0.9732
S,Fast-DetectGPT,0.7830,0.9667,0.6305,0.8292,0.7920,0.7986,0.6377,0.8729,0.8325
S,Binoculars,0.7603,0.9555,0.6211,0.7955,0.7889,0.7756,0.5985,0.8518,0.8075
S,LLM-Deviation,0.6843,0.8873,0.6233,0.6807,0.6584,0.6861,0.5610,0.7570,0.7088
P,BLOOMZ-3B-mixed-detector,0.6730,0.5410,0.6302,0.6655,0.6271,0.7694,0.6697,0.7000,0.6923
P,ChatGPT-detector-RoBERTa-Chinese,0.6423,0.7949,0.6784,0.6090,0.6906,0.5882,0.5667,0.6403,0.6711
P,Detection-Longformer,0.5615,0.6601,0.6045,0.5209,0.5045,0.4613,0.5453,0.6283,0.5879


\begin{tabular}{llrrrrrrrrr}
\bfseries Category & \bfseries Detector & \bfseries ALL_DATA & \bfseries Llama-2-70b-chat-hf & \bfseries Mistral-7B-Instruct-v0.2 & \bfseries aya-101 & \bfseries gemini & \bfseries gpt-3.5-turbo-0125 & \bfseries opt-iml-max-30b & \bfseries v5-Eagle-7B-HF & vicuna-13b \\
F & Llama-3.2-3B (de-pl-hu) & \bfseries {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9749 & {\cellcolor[HTML]{78ABD0}} \color[HTML]{000000} 0.9904 & {\cellcolor[HTML]{80AED2}} \color[HTML]{000000} 0.9688 & {\cellcolor[HTML]{81AED2}} \color[HTML]{000000} 0.9655 & \bfseries {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9754 & \bfseries {\cellcolor[HTML]{7BACD1}} \color[HTML]{000000} 0.9828 & \bfseries {\cellcolor[HTML]{83AFD3}} \color[HTML]{000000} 0.9632 & {\cellcolor[HTML]{7BACD1}} \color[HTML]{000000} 0.9822 & {\cellcolor[HTML]{7DACD1}} \color[HTML]{000000} 0.9789 \\
F & mDeBERTa-v3-base (de-pl-hr-hu-cs) & {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9734 & \bfseries {\cellcol

In [55]:
results_all = pd.DataFrame()
for dataset in [to_auc_dict(df[df.domain=="news"].reset_index(), True), to_auc_dict(df[df.domain=="social_media"].reset_index(), True)]:
 for detector in tqdm(selected):
  temp = pd.DataFrame({'Dataset': dataset['domains'][0], 'Detector': detector, 'Language': '{all}', 'AUC ROC': dataset[detector]['auc'], 'TPR @ 5% FPR': dataset[detector]['tpr_5%fpr'], 'Th @ 5% FPR': dataset[detector]['th_5%fpr'], 'Th @ max(TPR-FPR)': dataset[detector]['th_optim']}, index=[0])
  results_all = pd.concat([results_all, temp])
  for test_language,val in dataset[detector].items():
    if (test_language == 'auc') or ('_' in test_language): continue
    temp = pd.DataFrame({'Dataset': dataset['domains'][0], 'Detector': detector, 'Language': test_language, 'AUC ROC': dataset[detector][test_language]['auc'], 'TPR @ 5% FPR': val['tpr_5%fpr']}, index=[0])
    results_all = pd.concat([results_all, temp])
results_all = results_all[results_all['Detector'].isin(selected)]
#only best of each finetuned model version
temp = results_all.pivot_table(['AUC ROC', 'TPR @ 5% FPR'], 'Detector', ["Dataset", "Language"])['AUC ROC']#.reorder_levels(axis='columns', order=[1,0])#.sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp = temp.stack(level=0)
temp.rename(columns={'{all}':'All'}, inplace=True)
temp = temp.sort_index(axis='columns')
temp = temp.sort_values(by='All', ascending=False)
temp['temp'] = [x.split('_')[0] + y for x,y in zip(temp.reset_index()['Detector'], temp.reset_index()['Dataset'])]
temp = temp.drop_duplicates(subset='temp').drop(columns='temp')
#display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))

#temp = temp.reset_index()
#temp.index = temp.index.set_levels(pd.CategoricalIndex(temp.index.levels[1].values, categories=['news'], ordered=True), level=1)
#temp = temp.sort_index(level=1).sort_index(level=0, key=lambda x: x.map(adversarial_key))
temp = temp.reset_index()
temp['Detector'] = temp['Detector'].map(rename_detector)
temp = temp.set_index(['Dataset', 'Detector'])
temp = temp.sort_values(['Dataset', 'All'], ascending=False)
temp2 = temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).apply(lambda x: np.select([temp.eq(temp.groupby('Dataset').transform('max'))], ['font-weight: bold;'], default=''), axis=None).highlight_max(props='text-decoration: underline', axis=0).background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None)
display(temp2)
print(temp2.to_latex(convert_css=True).replace('\n\multirow','\n\hline\n\multirow'))

Best FPR: xlm-roberta-base_hr-cs sk 0.072
Best FPR: xlm-roberta-base_de-hr sl 0.076
Best FPR: xlm-roberta-base_pl de 0.088
Best FPR: xlm-roberta-base_hr sk 0.092
Best FPR: xlm-roberta-base_hr sl 0.292
Best FPR: xlm-roberta-base_hr-hu sk 0.076
Best FPR: xlm-roberta-base_hr-hu sl 0.052
Best FPR: mdeberta-v3-base_hr-cs pl 0.072
Best FPR: mdeberta-v3-base_de-hr-hu-cs pl 0.052
Best FPR: xlm-roberta-base_cs de 0.052
Best FPR: xlm-roberta-base_cs hr 0.068
Best FPR: xlm-roberta-base_cs sk 0.084
Best FPR: Llama-3.2-3B_hr sl 0.1
Best FPR: Llama-3.2-3B_de-pl-hr-hu hu 0.084
Best FPR: Llama-3.2-3B_de-pl-hr-hu sl 0.06
Best FPR: gemma-2-2b_pl-cs de 0.08
Best FPR: gemma-2-2b_pl de 0.256
Best FPR: gemma-2-2b_hr sl 0.096
Best FPR: gemma-2-2b_pl-hr de 0.088
Best FPR: gemma-2-2b_pl-hr-cs de 0.056
Best FPR: Llama-3.2-3B_de-hr-cs sl 0.052
Best FPR: andreas122001-bloomz-3b-mixed-detector sl 0.116
Best FPR: xlm-roberta-base_hr-cs cs 0.064
Best FPR: xlm-roberta-base_hr-cs de 0.14
Best FPR: xlm-roberta-base_hr-

100%|██████████| 130/130 [00:01<00:00, 115.87it/s]


\begin{tabular}{llrrrrrrrr}
 & Language & All & cs & de & hr & hu & pl & sk & sl \\
Dataset & Detector &  &  &  &  &  &  &  &  \\
\hline
\multirow[c]{10}{*}{social_media} & Llama-3.2-3B (de-pl-hu) & \bfseries {\cellcolor[HTML]{88B1D4}} \color[HTML]{000000} 0.9506 & \bfseries {\cellcolor[HTML]{7DACD1}} \color[HTML]{000000} 0.9800 & {\cellcolor[HTML]{8BB2D4}} \color[HTML]{000000} 0.9427 & \bfseries {\cellcolor[HTML]{83AFD3}} \color[HTML]{000000} 0.9628 & \bfseries {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9744 & \bfseries {\cellcolor[HTML]{89B1D4}} \color[HTML]{000000} 0.9466 & \bfseries {\cellcolor[HTML]{88B1D4}} \color[HTML]{000000} 0.9527 & {\cellcolor[HTML]{96B6D7}} \color[HTML]{000000} 0.9176 \\
 & mDeBERTa-v3-base (de) & {\cellcolor[HTML]{89B1D4}} \color[HTML]{000000} 0.9476 & {\cellcolor[HTML]{86B0D3}} \color[HTML]{000000} 0.9536 & \bfseries {\cellcolor[HTML]{88B1D4}} \color[HTML]{000000} 0.9515 & {\cellcolor[HTML]{88B1D4}} \color[HTML]{000000} 0.9503 & {\cellcolor[HTML]{8

In [ ]:
files = glob.glob("adversarial_*.csv.gz")
files

In [23]:
#load existing detectors
results_df = pd.DataFrame()
for file in tqdm(files):
  with gzip.open(file, 'rt') as gzf: #determine the index of last column
    reader = csv.reader(gzf, dialect=csv.excel)
    last_column_index = len(next(reader)) - 1
  if "Monoculars" in file:
    df = pd.read_csv(file)
    results_df[f"binoculars"] = pd.Series([x.replace('[','').replace(']','') for x in df['binoculars']]).astype(np.float16)
    results_df[f"fastdetectgpt"] = pd.Series([x.replace('[','').replace(']','') for x in df['fastgptdetect']]).astype(np.float16)
    results_df[f"llmdeviation"] = 1-pd.Series([x.replace('[','').replace(']','') for x in df['llmd1']]).astype(np.float16)
  else:
    df = pd.read_csv(file, usecols=[last_column_index], dtype=np.float16) #load only the last column
    model_name = file.split('/')[-1].split('adversarial_')[1].replace('_threshold', '').replace('.csv.gz', '').split('_')[-1]
    results_df[f"{model_name.replace('_','-')}"] = df[df.columns[-1]].astype(np.float16)
    if "unsup-simcse-roberta-base" == model_name: results_df[model_name] = 1-results_df[model_name]

100%|██████████| 4/4 [00:00<00:00, 47.41it/s]


In [24]:
df = pd.read_csv('adversarial_with_predictions.csv')

In [25]:
df.drop(columns=['text', 'split'], inplace=True)
df = pd.concat([df, results_df], axis=1)
selected = df.columns[4:].to_list()

In [26]:
df['obfuscator'] = [x.split('_')[-1] for x in df['multi_label']]
df.loc[(df.multi_label == "human_original"), 'obfuscator'] = "human_original"

In [27]:
df['obfuscator'].value_counts()

human_original    1400
original          1400
paraphrased       1400
homoglyph         1400
Name: obfuscator, dtype: int64

In [28]:
df.groupby('obfuscator').mean()

,label,is_machine_prob_mdeberta-v3-base,is_machine_prob_Llama-3.2-3B,is_machine_prob_xlm-roberta-base,is_machine_prob_gemma-2-2b,Hello-SimpleAI-chatgpt-detector-roberta-chinese,andreas122001-bloomz-3b-mixed-detector,binoculars,fastdetectgpt,llmdeviation,nealcly-detection-longformer
obfuscator,,,,,,,,,,,
homoglyph,1.0,0.428555,0.744980,0.290978,0.567209,0.087280,0.188110,-1.213867,-3.925781,-14.023438,0.615723
human_original,0.0,0.141293,0.097793,0.226763,0.063907,0.046539,0.232300,-1.133789,-1.230469,-13.710938,0.465332
original,1.0,0.944219,0.915888,0.909476,0.834483,0.143799,0.467529,-0.986816,0.472168,-7.375000,0.533691
paraphrased,1.0,0.972859,0.958918,0.936118,0.909662,0.155518,0.750000,-1.021484,-0.272461,-7.574219,0.474365


In [29]:
df.groupby(['obfuscator', 'language']).mean()

label  is_machine_prob_mdeberta-v3-base  \
obfuscator     language                                            
homoglyph      cs          1.0                          0.461934   
               de          1.0                          0.398483   
               hr          1.0                          0.419340   
               hu          1.0                          0.490703   
               pl          1.0                          0.373685   
               sk          1.0                          0.444450   
               sl          1.0                          0.411289   
human_original cs          0.0                          0.081430   
               de          0.0                          0.186075   
               hr          0.0                          0.121312   
               hu          0.0                          0.107270   
               pl          0.0                          0.154759   
               sk          0.0                          0.131674   
               sl          0.0                          0.206534   
original       cs          1.0                          0.925235   
               de          1.0                          0.958951   
               hr          1.0                          0.901760   
               hu          1.0                          0.974431   
               pl          1.0                          0.949426   
               sk          1.0                          0.930945   
               sl          1.0                          0.968785   
paraphrased    cs          1.0                          0.969491   
               de          1.0                          0.978511   
               hr          1.0                          0.938757   
               hu          1.0                          0.986432   
               pl          1.0                          0.973492   
               sk          1.0                          0.983746   
               sl          1.0                          0.979584   

                         is_machine_prob_Llama-3.2-3B  \
obfuscator     language                                 
homoglyph      cs                            0.862814   
               de                            0.739230   
               hr                            0.595987   
               hu                            0.866306   
               pl                            0.805376   
               sk                            0.742166   
               sl                            0.602980   
human_original cs                            0.105103   
               de                            0.098841   
               hr                            0.016797   
               hu                            0.121110   
               pl                            0.150570   
               sk                            0.118615   
               sl                            0.073518   
original       cs                            0.940765   
               de                            0.924182   
               hr                            0.867493   
               hu                            0.974227   
               pl                            0.950119   
               sk                            0.914067   
               sl                            0.840360   
paraphrased    cs                            0.977161   
               de                            0.969534   
               hr                            0.919137   
               hu                            0.981228   
               pl                            0.967263   
               sk                            0.973105   
               sl                            0.924999   

                         is_machine_prob_xlm-roberta-base  \
obfuscator     language                                     
homoglyph      cs                                0.329866   
               de                                0.221279   
               hr            

In [30]:
results_all = pd.DataFrame()
for dataset in [to_auc_dict(df)]:
 for detector in tqdm(selected):
  temp = pd.DataFrame({'Dataset': "ALL_DATA", 'Detector': detector, 'Language': '{all}', 'AUC ROC': dataset[detector]['auc'], 'TPR @ 5% FPR': dataset[detector]['tpr_5%fpr'], 'Th @ 5% FPR': dataset[detector]['th_5%fpr'], 'Th @ max(TPR-FPR)': dataset[detector]['th_optim']}, index=[0])
  results_all = pd.concat([results_all, temp])
  for test_language,val in dataset[detector].items():
    if (test_language == 'auc') or ('_' in test_language): continue
    temp = pd.DataFrame({'Dataset': test_language, 'Detector': detector, 'Language': test_language, 'AUC ROC': dataset[detector][test_language]['auc'], 'TPR @ 5% FPR': val['tpr_5%fpr']}, index=[0])
    results_all = pd.concat([results_all, temp])
results_all = results_all[results_all['Detector'].isin(selected)]
temp = results_all.pivot_table(['AUC ROC', 'TPR @ 5% FPR'], 'Detector', "Dataset").reorder_levels(axis='columns', order=[1,0]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
display(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.sort_values(by=('ALL_DATA','AUC ROC'), ascending=False).style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

100%|██████████| 10/10 [00:00<00:00, 106.30it/s]


\begin{tabular}{lrrrrrrrrrrrrrrrr}
Dataset & \multicolumn{2}{r}{ALL_DATA} & \multicolumn{2}{r}{cs} & \multicolumn{2}{r}{de} & \multicolumn{2}{r}{hr} & \multicolumn{2}{r}{hu} & \multicolumn{2}{r}{pl} & \multicolumn{2}{r}{sk} & \multicolumn{2}{r}{sl} \\
 & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR \\
Detector &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
is_machine_prob_Llama-3.2-3B & \bfseries 0.9648 & \bfseries 0.8214 & \bfseries 0.9779 & \bfseries 0.8983 & \bfseries 0.9595 & 0.7867 & \bfseries 0.9858 & \bfseries 0.9333 & \bfseries 0.9734 & \bfseries 0.8900 & \bfseries 0.9674 & \bfseries 0.8683 & \bfseries 0.9609 & \bfseries 0.7867 & \bfseries 0.9546 & \bfseries 0.7650 \\
is_machine_prob_gemma-2-2b & 0.9443 & 0.7562 & 0.9679 & 0.8417 & 0.9575 & \bfseries 0.8283 & 0.9534 & 0.8100 & 0.9707 & 0.8633 & 0.9565 & 0.7950 & 0.9507 & 0.

In [31]:
results_all = pd.DataFrame()
for obfuscator in df.obfuscator.unique():
 for dataset in [to_auc_dict(df[df.obfuscator.isin([obfuscator, 'human_original'])].reset_index())]:
  for detector in tqdm(selected):
   temp = pd.DataFrame({'Dataset': obfuscator, 'Detector': detector, 'Language': '{all}', 'AUC ROC': dataset[detector]['auc'], 'TPR @ 5% FPR': dataset[detector]['tpr_5%fpr'], 'Th @ 5% FPR': dataset[detector]['th_5%fpr'], 'Th @ max(TPR-FPR)': dataset[detector]['th_optim']}, index=[0])
   results_all = pd.concat([results_all, temp])
   for test_language,val in dataset[detector].items():
     if (test_language == 'auc') or ('_' in test_language): continue
     temp = pd.DataFrame({'Dataset': obfuscator, 'Detector': detector, 'Language': test_language, 'AUC ROC': dataset[detector][test_language]['auc'], 'TPR @ 5% FPR': val['tpr_5%fpr']}, index=[0])
     results_all = pd.concat([results_all, temp])
#results_all = results_all[results_all['Detector'].isin(selected)]
temp = results_all[results_all['Language'] == "{all}"].pivot_table(['AUC ROC', 'TPR @ 5% FPR'], 'Detector', "Dataset").reorder_levels(axis='columns', order=[1,0]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

c:\Users\User\anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:999: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
c:\Users\User\anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:999: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
c:\Users\User\anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:999: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
c:\Users\User\anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:999: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
c:\Users\User\anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:999: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
c:\Users\User\anaconda3\lib\site-packages\sklearn\metri

\begin{tabular}{lrrrrrrrr}
Dataset & \multicolumn{2}{r}{homoglyph} & \multicolumn{2}{r}{human_original} & \multicolumn{2}{r}{original} & \multicolumn{2}{r}{paraphrased} \\
 & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR & AUC ROC & TPR @ 5% FPR \\
Detector &  &  &  &  &  &  &  &  \\
Hello-SimpleAI-chatgpt-detector-roberta-chinese & 0.5547 & 0.0986 & \bfseries -1.0000 & \bfseries -1.0000 & 0.6497 & 0.1479 & 0.6515 & 0.1607 \\
andreas122001-bloomz-3b-mixed-detector & 0.4240 & 0.0379 & \bfseries -1.0000 & \bfseries -1.0000 & 0.6778 & 0.1657 & 0.8487 & 0.5036 \\
binoculars & 0.2711 & 0.0036 & \bfseries -1.0000 & \bfseries -1.0000 & 0.7758 & 0.4307 & 0.7098 & 0.2600 \\
fastdetectgpt & 0.0815 & 0.0014 & \bfseries -1.0000 & \bfseries -1.0000 & 0.8000 & 0.4400 & 0.7074 & 0.2464 \\
is_machine_prob_Llama-3.2-3B & \bfseries 0.9354 & \bfseries 0.6357 & \bfseries -1.0000 & \bfseries -1.0000 & \bfseries 0.9739 & \bfseries 0.8821 & \bfseries 0.9851 & \bfseries 0.9464 \\
is

In [32]:
print(results_all.Detector.unique().tolist())

['is_machine_prob_mdeberta-v3-base', 'is_machine_prob_Llama-3.2-3B', 'is_machine_prob_xlm-roberta-base', 'is_machine_prob_gemma-2-2b', 'Hello-SimpleAI-chatgpt-detector-roberta-chinese', 'andreas122001-bloomz-3b-mixed-detector', 'binoculars', 'fastdetectgpt', 'llmdeviation', 'nealcly-detection-longformer']


In [33]:
rename_adversarial_detector = {'is_machine_prob_Llama-3.2-3B': 'Llama-3.2-3B', 'is_machine_prob_mdeberta-v3-base': 'mDeBERTa-v3-base', 'is_machine_prob_gemma-2-2b': 'Gemma-2-2B', 'is_machine_prob_xlm-roberta-base': 'XLM-RoBERTa-base', 'fastdetectgpt': 'Fast-DetectGPT', 'binoculars': 'Binoculars', 'llmdeviation': 'LLM-Deviation', 'andreas122001-bloomz-3b-mixed-detector': 'BLOOMz-3B-mixed-detector', 'Hello-SimpleAI-chatgpt-detector-roberta-chinese': 'ChatGPT-detector-RoBERTa-Chinese', 'nealcly-detection-longformer': 'Detection-Longformer'}
adversarial_key = {v: k for k, v in dict(enumerate(rename_adversarial_detector.keys())).items()}

In [34]:
temp = results_all[results_all['Dataset'] != "human_original"].pivot_table('AUC ROC', 'Detector', ["Dataset", "Language"]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp = temp.stack(level=0)
temp.rename(columns={'{all}':'All'}, inplace=True)
temp.sort_index(axis=1, inplace=True)
temp.index = temp.index.set_levels(pd.CategoricalIndex(temp.index.levels[1].values, categories=['original', 'paraphrased', 'homoglyph'], ordered=True), level=1)
temp = temp.sort_index(level=1).sort_index(level=0, key=lambda x: x.map(adversarial_key))
temp = temp.reset_index()
temp['Detector'] = temp['Detector'].map(rename_adversarial_detector)
temp = temp.set_index(['Detector', 'Dataset'])
temp2 = temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).apply(lambda x: np.select([temp.eq(temp.groupby('Detector').transform('max'))], ['font-weight: bold;'], default=''), axis=None).highlight_max(props='text-decoration: underline', axis=0).background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None)
display(temp2)
print(temp2.to_latex(convert_css=True).replace('\n\multirow','\n\hline\n\multirow'))

\begin{tabular}{llrrrrrrrr}
 & Language & All & cs & de & hr & hu & pl & sk & sl \\
Detector & Dataset &  &  &  &  &  &  &  &  \\
\hline
\multirow[c]{3}{*}{Llama-3.2-3B} & original & {\cellcolor[HTML]{7EADD1}} \color[HTML]{000000} 0.9739 & {\cellcolor[HTML]{7DACD1}} \color[HTML]{000000} 0.9787 & {\cellcolor[HTML]{80AED2}} \color[HTML]{000000} 0.9717 & {\cellcolor[HTML]{79ABD0}} \color[HTML]{000000} 0.9881 & {\cellcolor[HTML]{7BACD1}} \color[HTML]{000000} 0.9825 & {\cellcolor[HTML]{7BACD1}} \color[HTML]{000000} 0.9808 & {\cellcolor[HTML]{80AED2}} \color[HTML]{000000} 0.9703 & {\cellcolor[HTML]{81AED2}} \color[HTML]{000000} 0.9654 \\
 & paraphrased & \bfseries \bfseries {\cellcolor[HTML]{79ABD0}} \color[HTML]{000000} 0.9851 & \bfseries \bfseries {\cellcolor[HTML]{78ABD0}} \color[HTML]{000000} 0.9921 & \bfseries {\cellcolor[HTML]{7DACD1}} \color[HTML]{000000} 0.9785 & \bfseries \bfseries {\cellcolor[HTML]{75A9CF}} \color[HTML]{000000} 0.9966 & \bfseries \bfseries {\cellcolor[HTML]{79ABD0}

In [35]:
from  matplotlib.colors import LinearSegmentedColormap
attack_cmap=LinearSegmentedColormap.from_list('rg',["r", "w"], N=256)

In [36]:
temp = results_all[results_all['Dataset'] != "human_original"].pivot_table('AUC ROC', 'Detector', ["Dataset", "Language"]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp2 = (temp - temp['original']) / temp['original'] * 100

temp2 = temp2.drop(columns='original').stack(level=0)
temp2.rename(columns={'{all}':'All'}, inplace=True)
temp2.sort_index(axis=1, inplace=True)
temp2.index = temp2.index.set_levels(pd.CategoricalIndex(temp2.index.levels[1].values, categories=['original', 'paraphrased', 'homoglyph'], ordered=True), level=1)
temp2 = temp2.sort_index(level=1).sort_index(level=0, key=lambda x: x.map(adversarial_key))
temp2 = temp2.reset_index()
temp2['Detector'] = temp2['Detector'].map(rename_adversarial_detector)
temp2 = temp2.set_index(['Detector', 'Dataset'])
temp3 = temp2.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).apply(lambda x: np.select([temp2.eq(temp2.groupby('Detector').transform('max'))], ['font-weight: bold;'], default=''), axis=None).highlight_max(props='text-decoration: underline', axis=0).background_gradient(cmap="OrRd_r", vmin=-150, vmax=0, text_color_threshold=0, axis=None)#.background_gradient(cmap="Greens", vmin=0, vmax=150, text_color_threshold=0, axis=None)
display(temp3)
print(temp3.to_latex(convert_css=True).replace('\n\multirow','\n\hline\n\multirow'))

\begin{tabular}{llrrrrrrrr}
 & Language & All & cs & de & hr & hu & pl & sk & sl \\
Detector & Dataset &  &  &  &  &  &  &  &  \\
\hline
\multirow[c]{2}{*}{Llama-3.2-3B} & paraphrased & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 1.1478 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 1.3756 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 0.6972 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 0.8615 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 0.4465 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 0.5965 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 1.4596 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 1.7855 \\
 & homoglyph & {\cellcolor[HTML]{FFF4E5}} \color[HTML]{000000} -3.9598 & {\cellcolor[HTML]{FFF6EA}} \color[HTML]{000000} -1.6017 & {\cellcolor[HTML]{FFF4E4}} \color[HTML]{000000} -4.4729 & {\cellcolor[HTML]{FFF6EA}} \color[HTML]{000000} -1.5661 & {\cellcolor[HTML]{FFF5E6}} \co

In [37]:
temp = results_all[results_all['Dataset'] != "human_original"].pivot_table('TPR @ 5% FPR', 'Detector', ["Dataset", "Language"]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp = temp.stack(level=0)
temp.rename(columns={'{all}':'All'}, inplace=True)
temp.sort_index(axis=1, inplace=True)
temp.index = temp.index.set_levels(pd.CategoricalIndex(temp.index.levels[1].values, categories=['original', 'paraphrased', 'homoglyph'], ordered=True), level=1)
temp = temp.sort_index(level=1).sort_index(level=0, key=lambda x: x.map(adversarial_key))
temp = temp.reset_index()
temp['Detector'] = temp['Detector'].map(rename_adversarial_detector)
temp = temp.set_index(['Detector', 'Dataset'])
temp2 = temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).apply(lambda x: np.select([temp.eq(temp.groupby('Detector').transform('max'))], ['font-weight: bold;'], default=''), axis=None).highlight_max(props='text-decoration: underline', axis=0).background_gradient(vmin=0.5, vmax=1.5, text_color_threshold=0, axis=None)
display(temp2)
print(temp2.to_latex(convert_css=True).replace('\n\multirow','\n\hline\n\multirow'))

\begin{tabular}{llrrrrrrrr}
 & Language & All & cs & de & hr & hu & pl & sk & sl \\
Detector & Dataset &  &  &  &  &  &  &  &  \\
\hline
\multirow[c]{3}{*}{Llama-3.2-3B} & original & {\cellcolor[HTML]{A4BCDA}} \color[HTML]{000000} 0.8821 & {\cellcolor[HTML]{94B6D7}} \color[HTML]{000000} 0.9200 & {\cellcolor[HTML]{A2BCDA}} \color[HTML]{000000} 0.8850 & {\cellcolor[HTML]{86B0D3}} \color[HTML]{000000} 0.9550 & {\cellcolor[HTML]{88B1D4}} \color[HTML]{000000} 0.9500 & {\cellcolor[HTML]{8FB4D6}} \color[HTML]{000000} 0.9300 & {\cellcolor[HTML]{B4C4DF}} \color[HTML]{000000} 0.8350 & {\cellcolor[HTML]{B1C2DE}} \color[HTML]{000000} 0.8400 \\
 & paraphrased & \bfseries \bfseries {\cellcolor[HTML]{89B1D4}} \color[HTML]{000000} 0.9464 & \bfseries \bfseries {\cellcolor[HTML]{80AED2}} \color[HTML]{000000} 0.9700 & \bfseries {\cellcolor[HTML]{8CB3D5}} \color[HTML]{000000} 0.9400 & \bfseries \bfseries {\cellcolor[HTML]{80AED2}} \color[HTML]{000000} 0.9700 & \bfseries \bfseries {\cellcolor[HTML]{80AED2}

In [38]:
temp = results_all[results_all['Dataset'] != "human_original"].pivot_table('TPR @ 5% FPR', 'Detector', ["Dataset", "Language"]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
temp2 = (temp - temp['original']) / temp['original'] * 100

temp2 = temp2.drop(columns='original').stack(level=0)
temp2.rename(columns={'{all}':'All'}, inplace=True)
temp2.sort_index(axis=1, inplace=True)
temp2.index = temp2.index.set_levels(pd.CategoricalIndex(temp2.index.levels[1].values, categories=['original', 'paraphrased', 'homoglyph'], ordered=True), level=1)
temp2 = temp2.sort_index(level=1).sort_index(level=0, key=lambda x: x.map(adversarial_key))
temp2 = temp2.reset_index()
temp2['Detector'] = temp2['Detector'].map(rename_adversarial_detector)
temp2 = temp2.set_index(['Detector', 'Dataset'])
temp3 = temp2.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).apply(lambda x: np.select([temp2.eq(temp2.groupby('Detector').transform('max'))], ['font-weight: bold;'], default=''), axis=None).highlight_max(props='text-decoration: underline', axis=0).background_gradient(cmap="OrRd_r", vmin=-150, vmax=0, text_color_threshold=0, axis=None)#.background_gradient(cmap="Greens", vmin=0, vmax=150, text_color_threshold=0, axis=None)
display(temp3)
print(temp3.to_latex(convert_css=True).replace('\n\multirow','\n\hline\n\multirow'))

\begin{tabular}{llrrrrrrrr}
 & Language & All & cs & de & hr & hu & pl & sk & sl \\
Detector & Dataset &  &  &  &  &  &  &  &  \\
\hline
\multirow[c]{2}{*}{Llama-3.2-3B} & paraphrased & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 7.2874 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 5.4348 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 6.2147 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 1.5707 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 2.1053 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 2.6882 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 12.5749 & \bfseries {\cellcolor[HTML]{FFF7EC}} \color[HTML]{000000} 9.5238 \\
 & homoglyph & {\cellcolor[HTML]{FEDFB4}} \color[HTML]{000000} -27.9352 & {\cellcolor[HTML]{FEEDD4}} \color[HTML]{000000} -12.5000 & {\cellcolor[HTML]{FDD19B}} \color[HTML]{000000} -39.5480 & {\cellcolor[HTML]{FFF0DC}} \color[HTML]{000000} -8.3770 & {\cellcolor[HTML]{FEE6C4}}

In [15]:
adversarial=['original', 'paraphrased', 'homoglyph']
adversarial_key = {v: k for k, v in dict(enumerate(adversarial)).items()}

In [110]:
temp = results_all[results_all['Dataset'] != "human_original"].pivot_table('TPR @ 5% FPR', 'Detector', ["Dataset", "Language"]).sort_index(axis='columns')#.sort_index(axis='index', ascending=False)
display(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0))
print(temp.style.format(precision=4).highlight_max(props='font-weight: bold;', axis=0).to_latex(convert_css=True))

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrr}
Dataset & \multicolumn{8}{r}{homoglyph} & \multicolumn{8}{r}{original} & \multicolumn{8}{r}{paraphrased} \\
Language & cs & de & hr & hu & pl & sk & sl & {all} & cs & de & hr & hu & pl & sk & sl & {all} & cs & de & hr & hu & pl & sk & sl & {all} \\
Detector &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
Hello-SimpleAI-chatgpt-detector-roberta-chinese & 0.1200 & 0.0300 & 0.1600 & 0.1550 & 0.1250 & 0.2100 & 0.1400 & 0.0986 & 0.1750 & 0.1850 & 0.2050 & 0.1950 & 0.1850 & 0.2750 & 0.1900 & 0.1479 & 0.1300 & 0.1300 & 0.2250 & 0.2700 & 0.2000 & 0.2950 & 0.1450 & 0.1607 \\
andreas122001-bloomz-3b-mixed-detector & 0.0500 & 0.0200 & 0.0500 & 0.0650 & 0.0500 & 0.0600 & 0.0050 & 0.0379 & 0.1850 & 0.1600 & 0.2250 & 0.2200 & 0.1300 & 0.2150 & 0.1250 & 0.1657 & 0.5250 & 0.4250 & 0.5550 & 0.5350 & 0.5150 & 0.5600 & 0.4100 & 0.5036 \\
binoculars & 0.0050 & 0.0000 & 0.0050 & 0.0000 & 0.0100 & 0.0050 & 0.0050 & 0.0036 & 0.5150 & 0.495

In [85]:
temp2 = (temp - temp['original']) / temp['original'] * 100
display(temp2.drop(columns='original').stack(level=0).style.format(precision=2).highlight_min(props='font-weight: bold;', axis=0))

In [3]:
df = pd.read_feather('multidomain_test_ce_subset_adversarial.feather')

In [4]:
df['obfuscator'] = [x.split('_')[-1] for x in df['multi_label']]

In [7]:
temp = df.groupby(['obfuscator'])[['meteor', 'bertscore', 'ngram', 'tf', 'ED-norm', 'diff_charlen', 'changed_language']].agg(['mean', 'std'])
#temp['changed_language'] = len(df[df.language != df.fasttext]) / len(df)
temp.drop(columns=('changed_language', 'std'), inplace=True)
temp.loc[['paraphrased', 'homoglyph'], :]

meteor          bertscore               ngram            \
                 mean      std      mean       std      mean       std   
obfuscator                                                               
paraphrased  0.477070  0.22984  0.823716  0.085168  0.428555  0.198052   
homoglyph    0.215982  0.23831  0.872342  0.053368  0.618921  0.082184   

                   tf             ED-norm            diff_charlen             \
                 mean       std      mean        std         mean        std   
obfuscator                                                                     
paraphrased  0.683546  0.224773  1.488706  16.150951     1.988181  16.175087   
homoglyph    0.311312  0.247448  0.099092   0.029065     1.060717   0.348781   

            changed_language  
                        mean  
obfuscator                    
paraphrased         0.176429  
homoglyph           0.146429

In [14]:
temp = temp.loc[['paraphrased', 'homoglyph'], :].copy()
temp2 = temp.copy()
for col in temp.columns.get_level_values(0).unique()[:-1]:
  temp2[col] = [f"{str('%.3f' % x)} (±{str('%.2f' % y)})" for x,y in zip(temp[(col, 'mean')], temp[(col, 'std')])]
temp2.columns = temp2.columns.droplevel(level=1)
temp2 = temp2.T.drop_duplicates().T
#temp2.rename(index=rename_obfuscators, inplace=True)
temp2.columns = ['METEOR', 'BERTScore', 'ngram', 'TF', 'LD', 'CharLenDiff', 'LangCheck']
temp2.T.style.highlight_max(props='font-weight: bold;', axis=1).format({'LangCheck': '{:,.2%}'.format}, na_rep=0, precision=4)

obfuscator,paraphrased,homoglyph
METEOR,0.477 (±0.23),0.216 (±0.24)
BERTScore,0.824 (±0.09),0.872 (±0.05)
ngram,0.429 (±0.20),0.619 (±0.08)
TF,0.684 (±0.22),0.311 (±0.25)
LD,1.489 (±16.15),0.099 (±0.03)
CharLenDiff,1.988 (±16.18),1.061 (±0.35)
LangCheck,0.1764,0.1464


In [16]:
print(temp2.T.style.highlight_max(props='font-weight: bold;', axis=1).format({'LangCheck': '{:,.2%}'.format}, na_rep=0, precision=4).applymap_index(lambda v: "font-weight: bold;", axis=0).applymap_index(lambda v: "font-weight: bold;", axis=1).to_latex(convert_css=True).replace('%', '\%'))

\begin{tabular}{lll}
obfuscator & \bfseries paraphrased & \bfseries homoglyph \\
\bfseries METEOR & \bfseries 0.477 (±0.23) & 0.216 (±0.24) \\
\bfseries BERTScore & 0.824 (±0.09) & \bfseries 0.872 (±0.05) \\
\bfseries ngram & 0.429 (±0.20) & \bfseries 0.619 (±0.08) \\
\bfseries TF & \bfseries 0.684 (±0.22) & 0.311 (±0.25) \\
\bfseries LD & \bfseries 1.489 (±16.15) & 0.099 (±0.03) \\
\bfseries CharLenDiff & \bfseries 1.988 (±16.18) & 1.061 (±0.35) \\
\bfseries LangCheck & \bfseries 0.1764 & 0.1464 \\
\end{tabular}

